In [1]:
import sys
from pathlib import Path

# Notebook cwd is notebooks/, so add project root for local package imports.
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ingest import *
from src.aggregate import *
from src.utils import *
from src.output import *

In [ ]:
country_list = list(pd.read_csv(DATA_DIR / "import_log.csv")["country"].unique())
country_list = list(country_list)

In [29]:
type(country_list)


list

In [2]:
from pathlib import Path

context = get_project_context(start_path=Path.cwd().resolve())
CONFIG_DIR = context["config_dir"]
OUTPUT_DIR = context["output_dir"]
DATA_DIR = context["data_dir"]
INPUT_DIR = context["input_dir"]

In [3]:
filepath = str((INPUT_DIR / "aser_sample.csv").resolve())
dataset_aser = build_dataset(file_path=filepath)

Form ID aser_001 already exists in the log. Processed data will be added to the existing dataset.
Metadata added to log:
{'form_id': 'aser_001', 'survey_name': 'aser_kenya', 'level': 'child', 'first_submission': '2021-01-02 00:00:00', 'last_submission': '2021-01-22 00:00:00', 'rows': 200, 'country': 'kenya', '_uuid_cols': np.float64(nan)}
Data imported from file. Kobo metadata functions will be skipped (question dataframe and choices dataframe). Only basic metadata will be logged.
Dropped columns: ['child_name'] to protect child privacy


In [4]:
log = pd.read_csv(DATA_DIR / "import_log.csv")

In [5]:
log.head()

,form_id,survey_name,level,first_submission,last_submission,rows,country,_uuid_cols,logged_at
0,aser_001,aser_kenya,child,2021-01-02 00:00:00,2021-01-22 00:00:00,200,kenya,NaN,2026-06-13 10:13:00
1,haldo_001,Haldo survey,child,2022-02-02 00:00:00,NaN,1563,kenya,NaN,2026-06-14 06:36:00
2,aser_001,aser_kenya,child,2021-01-02 00:00:00,2021-01-22 00:00:00,200,kenya,NaN,2026-06-14 06:37:00
3,aser_01,aser_sample,child,NaN,NaN,200,Kenya,NaN,2026-06-14 06:45:00
4,aser_01,aser_sample,child,NaN,NaN,200,Kenya,NaN,2026-06-14 06:45:00


In [23]:
filepath = str((INPUT_DIR / "haldo_sample.xlsx").resolve())
dataset_haldo = build_dataset(file_path=filepath)

Form ID haldo_001 already exists in the log. Processed data will be added to the existing dataset.
Metadata added to log:
{'form_id': 'haldo_001', 'survey_name': 'Haldo survey', 'level': 'child', 'first_submission': '2022-02-02 00:00:00', 'last_submission': nan, 'rows': 1246, 'country': 'kenya', '_uuid_cols': np.float64(nan)}
Data imported from file. Kobo metadata functions will be skipped (question dataframe and choices dataframe). Only basic metadata will be logged.


In [7]:
dataset_aser.df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 27 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   round                  200 non-null    str  
 1   enumerator             200 non-null    str  
 2   school_name            200 non-null    str  
 3   child_uid              200 non-null    str  
 4   child_sex              200 non-null    str  
 5   child_age              200 non-null    int64
 6   school_status          200 non-null    str  
 7   child_grade            200 non-null    int64
 8   disability_status      200 non-null    str  
 9   protection_status      200 non-null    str  
 10  language_match         200 non-null    str  
 11  location               200 non-null    str  
 12  region                 200 non-null    str  
 13  reading_score          200 non-null    int64
 14  reading_word           200 non-null    int64
 15  reading_letter         200 non-null    int64
 16  r

In [24]:
dataset_haldo.df['_uuid_cols']

0       start,end,childid
1       start,end,childid
2       start,end,childid
3       start,end,childid
4       start,end,childid
              ...        
1241    start,end,childid
1242    start,end,childid
1243    start,end,childid
1244    start,end,childid
1245    start,end,childid
Name: _uuid_cols, Length: 1246, dtype: str

In [9]:
ds_aser = aggregates(dataset_aser)
ds_haldo = aggregates(dataset_haldo)

Available dimensions: ['admin1', 'location', 'protection_status', 'school', 'sex', 'age', 'disability', 'age_group']
Missing dimensions: ['admin0', 'admin2', 'admin3', 'grade']
Resolved columns: {'admin0': None, 'admin1': 'region', 'admin2': None, 'admin3': None, 'location': 'location', 'protection_status': 'protection_status', 'school': 'school_name', 'sex': 'child_sex', 'age': 'child_age', 'disability': 'disability_status', 'grade': None, 'age_group': 'child_age'}
Derived dimensions computed: ['age_group']
Available indicators: ['aser_literacy', 'aser_numeracy']
Missing indicators: ['haldo_self_concept', 'haldo_empathy', 'haldo_expressive_language', 'haldo_letter_identification', 'haldo_accuracy', 'haldo_comprehension', 'haldo_sel', 'haldo_literacy']
Available dimensions: ['admin1', 'admin2', 'admin3', 'school', 'sex', 'age', 'grade', 'age_group']
Missing dimensions: ['admin0', 'location', 'protection_status', 'disability']
Resolved columns: {'admin0': None, 'admin1': 'governorate', 

In [10]:
#==============================================
# Build child level combined dataset
#==============================================

def build_combined_df(dataset, OUTPUT_DIR=OUTPUT_DIR):
    country = dataset.metadata["country"]
    level = dataset.metadata["level"]
    output_dir = OUTPUT_DIR / country / "tables"
    df = dataset.df_short.copy()
    df = df.rename(columns={"uid": f"{level}_uid"})
    output_path = output_dir / f"combined_{level}_level.csv"

    if output_path.exists():
        df_combined = pd.read_csv(output_path)
        df = pd.concat([df_combined, df], ignore_index=True)
        df = df.drop_duplicates(subset="_uuid")

    if not output_dir.exists():
        output_dir.mkdir(parents=True, exist_ok=True)

    df.to_csv(output_path, index=False)

    return df
    

In [13]:
comb_child = build_combined_df(dataset_haldo, OUTPUT_DIR)

In [15]:
comb_child.info()

<class 'pandas.DataFrame'>
RangeIndex: 2363 entries, 0 to 2362
Data columns (total 22 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   admin1                       2046 non-null   object 
 1   location                     800 non-null    str    
 2   protection_status            800 non-null    str    
 3   school                       1846 non-null   str    
 4   sex                          2046 non-null   object 
 5   age                          2046 non-null   float64
 6   disability                   800 non-null    str    
 7   age_group                    2046 non-null   str    
 8   aser_literacy                800 non-null    float64
 9   aser_numeracy                800 non-null    float64
 10  _uuid                        2363 non-null   str    
 11  admin2                       1246 non-null   float64
 12  admin3                       1246 non-null   float64
 13  grade                        